# pyct smoke test

`pyct` (Python Config Tools) is a small helper library used by HoloViz projects to wire up example-fetching CLIs, report package versions, and manage build/version metadata. It is **not** the `param` library — it exposes `pyct.report`, `pyct.cmd`, and `pyct.build`.

In [ ]:
# Example 1: pyct.report() prints a version + install-location line per package.
import contextlib
import io

import pyct

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    pyct.report('pyct')
out = buf.getvalue().strip()
print(out)

# The report line names the package and its resolved version.
assert 'pyct' in out
assert pyct.__version__ in out
assert pyct.NAME == 'pyct'

In [ ]:
# Example 2: the pyct.cmd CLI-helper surface exposes the documented entry points.
import pyct.cmd as cmd

expected = ['copy_examples', 'fetch_data', 'examples', 'clean_data',
            'substitute_main', 'add_commands', 'add_version', 'Bar', 'DirectoryContext']
present = [name for name in expected if hasattr(cmd, name)]
print(f"present: {present}")

for name in expected:
    assert hasattr(cmd, name), f"pyct.cmd is missing {name!r}"
# The example-management commands are callables.
for name in ('copy_examples', 'fetch_data', 'examples', 'clean_data'):
    assert callable(getattr(cmd, name))

In [ ]:
# Example 3: DirectoryContext temporarily chdir's into a path, then restores cwd.
import os
import tempfile

from pyct.cmd import DirectoryContext

before = os.getcwd()
with tempfile.TemporaryDirectory() as tmp:
    target = os.path.realpath(tmp)
    with DirectoryContext(tmp):
        inside = os.path.realpath(os.getcwd())
        print(f"inside context: {inside}")
        assert inside == target
    # cwd is restored on exit.
    print(f"after context: {os.getcwd()}")
    assert os.getcwd() == before

In [ ]:
# Example 4: pyct.build exposes version-derivation helpers used at packaging time.
import inspect

import pyct.build as build

public = [n for n in dir(build) if not n.startswith('_')]
print(f"pyct.build public names: {public}")

assert hasattr(build, 'get_setup_version')
sig = inspect.signature(build.get_setup_version)
print(f"get_setup_version{sig}")
# It takes a project root + repo name to compute a version string.
assert list(sig.parameters) == ['root', 'reponame']

In [ ]:
# Example 5: the Bar progress-bar helper instantiates + carries its expected state.
from pyct.cmd import Bar

bar = Bar(label='work', expected_size=10)
attrs = [n for n in dir(bar) if not n.startswith('_')]
print(f"Bar attributes: {attrs}")

assert bar.expected_size == 10
assert bar.label == 'work'
# Rendering primitives are present (filled/empty glyphs + width).
for attr in ('filled_char', 'empty_char', 'width', 'show', 'done'):
    assert hasattr(bar, attr), f"Bar missing {attr!r}"
print("pyct smoke test PASS")